In [ ]:
from IPython.display import clear_output
from Fetch_Live_Data import *
from Trade_Selection_All import *
from Trade_Execution import *
from Chandelier_ZLSMA_Filter import *

In [ ]:
def chandelier_zlsma_filter(
    timeframe: str = '15m',
    scan_limit: int = 200,
    max_buy_candles: int = 3,
    zlsma_length: int = 200,
    zlsma_limit: int = 500
) -> list:
    # 1. Initialize scanner and load markets
    scanner = BinanceChandelierScanner()
    scanner.exchange.load_markets()

    # 2. Scan for chandelier buy signals
    raw_results = scanner.scan_pairs(
        timeframe=timeframe,
        limit=scan_limit,
        max_buy_candles=max_buy_candles
    )
    results_df = pd.DataFrame(raw_results)

    # 3. Extract symbols and filter by ZLSMA
    symbols = results_df['symbol'].tolist()
    filtered_coins = filter_coins_above_zlsma(
        symbols,
        length=zlsma_length,
        timeframe=timeframe,
        limit=zlsma_limit
    )

    return filtered_coins

In [ ]:
out = chandelier_zlsma_filter()
out

In [22]:

def pick_best_coin():
    scalping_filter = BinanceAllUSDTScalpingFilter(
        max_workers=8,
        delay_between_requests=0.1,
        weight_profile='volatile'  # Options: 'balanced', 'volatile', 'trending'
    )

    print("Scanning ALL Binance USDT pairs for scalping opportunities...")

    # get a list of dicts (or records)
    best_coins = scalping_filter.filter_all_usdt_pairs(
        min_volume=50_000,
        top_n=30,
        use_parallel=True,
        volume_filter_first=False,
        use_percentile_volume=True
    )

    # turn into a DataFrame
    df = pd.DataFrame(best_coins)
    df = df[df['current_price'] <= 50]

    # base filters: uptrend + cheap coins
    #base_mask = (df['trend_direction'] == 1) & (df['current_price'] <= 50)

    # nothing matched
    return df["symbol"].tolist()

In [ ]:
# Define your function to fetch, calculate, and merge the data
def fetch_and_process_data(symbol):
    df = get_single_fetch(symbol, 500)  # Fetch data
    df_zlsma = calculate_zlsma(df, close_column='close', length=200)  # Calculate ZLSMA
    df_chandelier = calculate_chandelier(df, atr_period=1, atr_multiplier=2.0)  # Calculate Chandelier
    merged_chandelier_zlsma = merge_zlsma_chandelier(df_zlsma, df_chandelier)  # Merge ZLSMA and Chandelier
    merged_chandelier_zlsma = merged_chandelier_zlsma[["timestamp", "close", "zlsma_200", "buy_signal", "sell_signal"]]  # Filter relevant columns
    return merged_chandelier_zlsma[480:]

def run_task():
    # Get the latest symbols every 2 hours
    symbols = pick_best_coin()
    return symbols

def fetch_and_display_data_(symbols):
    # Run the fetch_and_process_data every 10 seconds to display results
    result = fetch_and_process_data(symbols)
    clear_output(wait=True)  # Clear previous output in Jupyter Notebook
    print("Monitoring on: ", symbols)
    print(result)  # Display the new result
    print("\n")
    
    return result

# Main loop that runs every 2 hours
while True:
    symbols = run_task()  # Get the symbols every 2 hours (list of up to 10 symbols)
    print("New symbols received. Monitoring begins...\n")
    print(f"Symbols to monitor ({len(symbols)} symbols): {symbols}")
    
    trade_taken = False  # Flag to track if any trade was taken
    
    # Try each symbol one by one
    for i, symbol in enumerate(symbols):
        print(f"\n--- Checking symbol {i+1}/{len(symbols)}: {symbol} ---")
        
        # Continuous 10-second updates with fetched data for current symbol
        symbol_checked = False
        
        while not symbol_checked:
            out = fetch_and_display_data_([symbol])  # Pass single symbol as list
            
            # Check if the DataFrame is not empty and get the last row
            if not out.empty:
                # First condition: Check if buy_signal sum is <= 15
                if out['buy_signal'].sum() <= 15:
                    print(f"Buy signal sum condition met for {symbol}: {out['buy_signal'].sum()}")
                    
                    # Second condition: Check price above ZLSMA and buy signal
                    if out['close'].iloc[-1] > out['zlsma_200'].iloc[-1] and out['buy_signal'].iloc[-1] == True:
                        print(f"✅ All conditions met! Taking trade for: {symbol}")
                        #bot = SimpleATRTradingBot()
                        #result = bot.buy_signal(symbol, 10)
                        #status = bot.get_position_status()
                        print(f"✅ Trade Taken: {symbol}")
                        print("Running trade for 2 hours...")
                        trade_taken = True
                        symbol_checked = True  # Move to next phase
                        break  # Exit the symbol checking loop
                    else:
                        print(f"Buy signal sum ok but no buy signal detected for {symbol}")
                        symbol_checked = True  # Try next symbol
                else:
                    print(f"Buy signal sum > 15 for {symbol} (sum: {out['buy_signal'].sum()}), trying next symbol...")
                    symbol_checked = True  # Try next symbol
            else:
                print(f"The DataFrame is empty for {symbol}. No data available.")
                symbol_checked = True  # Try next symbol
        
        # If trade was taken, break out of symbol iteration
        if trade_taken:
            break
    
    # If no trade was taken with any symbol, refresh for new symbols
    if not trade_taken:
        print("\n❌ No trades taken with any symbols. Refreshing for new symbols...")
        continue  # Skip the 2-hour sleep and get new symbols immediately
    
    # If trade was taken, monitor for 2 hours with 10-second intervals
    print(f"\n🔄 Monitoring trade for 2 hours...")
    monitoring_start = time.time()
    
    while time.time() - monitoring_start < 2 * 60 * 60:  # 2 hours
        # You can add monitoring logic here if needed
        time.sleep(10)  # Wait for 10 seconds before next check
    
    print("2-hour monitoring period completed. Getting new symbols...")

In [24]:
import time
import logging
import pandas as pd
import signal
import sys
from typing import List, Optional
import configparser
from IPython.display import clear_output

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('trading_bot.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Load configuration
config = configparser.ConfigParser()
config.read('config.ini')
MONITORING_INTERVAL = config.getint('Settings', 'monitoring_interval', fallback=10)  # seconds
CYCLE_DURATION = config.getint('Settings', 'cycle_duration', fallback=2 * 60 * 60)  # 2 hours
BUY_SIGNAL_THRESHOLD = config.getint('Settings', 'buy_signal_threshold', fallback=15)

# Graceful shutdown
running = True

def signal_handler(sig, frame):
    global running
    logger.info("Shutdown signal received. Exiting gracefully...")
    running = False
    sys.exit(0)

signal.signal(signal.SIGINT, signal_handler)
signal.signal(signal.SIGTERM, signal_handler)

def fetch_and_process_data(symbol: str) -> Optional[pd.DataFrame]:
    """Fetch and process data for a given symbol."""
    try:
        df = get_single_fetch(symbol, 500)  # Fetch data
        if df.empty:
            logger.warning(f"No data fetched for {symbol}")
            return pd.DataFrame()

        df_zlsma = calculate_zlsma(df, close_column='close', length=200)  # Calculate ZLSMA
        df_chandelier = calculate_chandelier(df, atr_period=1, atr_multiplier=2.0)  # Calculate Chandelier
        merged_chandelier_zlsma = merge_zlsma_chandelier(df_zlsma, df_chandelier)  # Merge ZLSMA and Chandelier
        required_columns = ["timestamp", "close", "zlsma_200", "buy_signal", "sell_signal"]
        if not all(col in merged_chandelier_zlsma.columns for col in required_columns):
            logger.error(f"Missing required columns in DataFrame for {symbol}: {merged_chandelier_zlsma.columns}")
            return pd.DataFrame()

        result = merged_chandelier_zlsma[required_columns][480:]
        logger.debug(f"Processed data for {symbol}: {len(result)} rows")
        return result
    except Exception as e:
        logger.error(f"Error processing data for {symbol}: {e}")
        return pd.DataFrame()

def run_task() -> Optional[List[str]]:
    """Fetch symbols to monitor."""
    try:
        symbols = pick_best_coin()
        if not isinstance(symbols, list):
            logger.error(f"Invalid symbols list: {symbols}")
            return []
        logger.info(f"Fetched symbols: {symbols}")
        return symbols
    except Exception as e:
        logger.error(f"Error in run_task: {e}")
        return []

def fetch_and_display_data(symbols: List[str]) -> Optional[pd.DataFrame]:
    """Fetch and display data for given symbols."""
    try:
        if len(symbols) != 1:
            logger.error(f"Expected single symbol, got {symbols}")
            return pd.DataFrame()

        symbol = symbols[0]
        result = fetch_and_process_data(symbol)
        if result.empty:
            logger.warning(f"No data available for {symbol}")
            return result

        clear_output(wait=True)
        logger.info(f"Monitoring on: {symbol}")
        print(f"Monitoring on: {symbol}")
        print(result)
        print("\n")
        return result
    except Exception as e:
        logger.error(f"Error in fetch_and_display_data for {symbols}: {e}")
        return pd.DataFrame()

def main():
    global running
    while running:
        symbols = run_task()
        if symbols is None or not symbols:
            logger.warning("No symbols received. Retrying in 60 seconds...")
            time.sleep(60)
            continue

        logger.info(f"New symbols received. Monitoring begins... ({len(symbols)} symbols: {symbols})")
        trade_taken = False

        for i, symbol in enumerate(symbols):
            if not running:
                break
            logger.info(f"Checking symbol {i+1}/{len(symbols)}: {symbol}")

            symbol_checked = False
            while not symbol_checked and running:
                out = fetch_and_display_data([symbol])
                if out.empty:
                    logger.warning(f"No data available for {symbol}. Moving to next symbol.")
                    symbol_checked = True
                    continue

                buy_signal_sum = out['buy_signal'].sum()
                if buy_signal_sum <= BUY_SIGNAL_THRESHOLD:
                    logger.info(f"Buy signal sum condition met for {symbol}: {buy_signal_sum}")
                    last_row = out.iloc[-1]
                    if last_row['close'] > last_row['zlsma_200'] and last_row['buy_signal']:
                        logger.info(f"✅ All conditions met! Taking trade for: {symbol}")
                        try:
                            # Uncomment and implement actual trading logic
                            # bot = SimpleATRTradingBot()
                            # result = bot.buy_signal(symbol, 10)
                            # status = bot.get_position_status()
                            logger.info(f"✅ Trade Taken: {symbol}")
                            logger.info("Running trade for 2 hours...")
                            trade_taken = True
                            symbol_checked = True
                            break
                        except Exception as e:
                            logger.error(f"Error executing trade for {symbol}: {e}")
                            symbol_checked = True
                    else:
                        logger.info(f"Buy signal sum ok but no buy signal detected for {symbol}")
                        symbol_checked = True
                else:
                    logger.info(f"Buy signal sum > {BUY_SIGNAL_THRESHOLD} for {symbol} (sum: {buy_signal_sum}). Trying next symbol.")
                    symbol_checked = True

                time.sleep(MONITORING_INTERVAL)

            if trade_taken:
                break

        if not trade_taken:
            logger.info("No trades taken. Refreshing for new symbols...")
            continue

        logger.info("Monitoring trade for 2 hours...")
        monitoring_start = time.time()
        while running and (time.time() - monitoring_start < CYCLE_DURATION):
            # Add trade monitoring logic here
            time.sleep(MONITORING_INTERVAL)

        logger.info("2-hour monitoring period completed. Getting new symbols...")

In [25]:
if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        logger.critical(f"Fatal error: {e}")
        sys.exit(1)

2025-06-14 13:08:47,330 - INFO - Monitoring on: XRPUSDT
2025-06-14 13:08:47,331 - INFO - Buy signal sum condition met for XRPUSDT: 11
2025-06-14 13:08:47,332 - INFO - ✅ All conditions met! Taking trade for: XRPUSDT
2025-06-14 13:08:47,332 - INFO - ✅ Trade Taken: XRPUSDT
2025-06-14 13:08:47,332 - INFO - Running trade for 2 hours...
2025-06-14 13:08:47,332 - INFO - Monitoring trade for 2 hours...


Monitoring on: XRPUSDT
              timestamp   close  zlsma_200  buy_signal  sell_signal
480 2025-06-14 08:15:00  2.1477   2.127139           1            0
481 2025-06-14 08:30:00  2.1498   2.128409           1            0
482 2025-06-14 08:45:00  2.1472   2.129408           1            0
483 2025-06-14 09:00:00  2.1438   2.130211           1            0
484 2025-06-14 09:15:00  2.1410   2.130733           1            0
485 2025-06-14 09:30:00  2.1436   2.131364           1            0
486 2025-06-14 09:45:00  2.1399   2.131905           1            0
487 2025-06-14 10:00:00  2.1344   2.132316           0            1
488 2025-06-14 10:15:00  2.1376   2.132888           0            1
489 2025-06-14 10:30:00  2.1386   2.133590           0            1
490 2025-06-14 10:45:00  2.1397   2.134365           0            1
491 2025-06-14 11:00:00  2.1430   2.135150           0            1
492 2025-06-14 11:15:00  2.1427   2.135944           0            1
493 2025-06-14 11:30:00  

KeyboardInterrupt: 

In [ ]:
bot = SimpleATRTradingBot()
result = bot.buy_signal('BTC', 1000)
status = bot.get_position_status()